## Ejercicio 2: Escalamiento de tickets de soporte técnico

### Ficha PEAS
| Elemento | Descripción |
| :--- | :--- |
| **Percepción (S)** | Tiempo de espera en minutos (`tiempo_espera_minutos`), nivel de urgencia (`nivel_urgencia`: "baja", "media", "alta") y tipo de cliente (`cliente_premium`: True/False). |
| **Acciones (A)** | Asignar ticket a Nivel 1 (soporte básico), Nivel 2 (soporte técnico especializado) o Nivel 3 (atención crítica/prioritaria). |
| **Entorno (E)** | Sistema de mesa de ayuda (Help Desk) y cola de atención de soporte técnico. |
| **Objetivo** | Optimizar los tiempos de resolución y cumplir con los acuerdos de nivel de servicio (SLA) según el perfil del cliente. |
| **Medida de Desempeño (P)** | Tiempo medio de resolución (MTTR), tasa de satisfacción del cliente (CSAT) y cumplimiento de tiempos de respuesta por SLA. |

### Justificación de las reglas
El esquema de reglas garantiza que las urgencias altas se deriven de inmediato a Nivel 3 para contener fallas críticas sin demoras. Ademas, compensa la paciencia del cliente priorizando a usuarios VIP mediante un umbral de tiempo reducido (30 min vs. 60 min para escalamiento), asegurando así un uso eficiente de los recursos del equipo de soporte sin descuidar los compromisos comerciales de mayor valor.

### Implementación del Agente

In [1]:
def agente_soporte(tiempo_espera_minutos, nivel_urgencia, cliente_premium):
    # Regla 1: Urgencia alta se escala directamente a Nivel 3
    if nivel_urgencia == "alta":
        return "escalar a nivel 3", "Ticket clasificado con urgencia alta que requiere intervención crítica"
    
    # Regla 2: Urgencia media depende de la condición de cliente y tiempo de espera
    elif nivel_urgencia == "media":
        if cliente_premium or tiempo_espera_minutos > 45:
            return "escalar a nivel 2", f"Urgencia media con factor prioritario (Premium={cliente_premium}, Espera={tiempo_espera_minutos} min)"
        else:
            return "asignar a nivel 1", f"Urgencia media estándar con tiempo de espera aceptable ({tiempo_espera_minutos} min)"
            
    # Regla 3: Urgencia baja se evalúa principalmente por acumulación de tiempo
    elif nivel_urgencia == "baja":
        if cliente_premium and tiempo_espera_minutos > 30:
            return "escalar a nivel 2", f"Cliente premium con espera prolongada ({tiempo_espera_minutos} min) para urgencia baja"
        elif tiempo_espera_minutos > 60:
            return "escalar a nivel 2", f"Tiempo de espera crítico en cola ({tiempo_espera_minutos} min) requiere escalamiento"
        else:
            return "asignar a nivel 1", f"Ticket de rutina en tiempo tolerable ({tiempo_espera_minutos} min)"
            
    else:
        return "asignar a nivel 1", "Nivel de urgencia no identificado, asignado a soporte básico por defecto"

### Simulación y Pruebas

In [2]:
# Definición de 6 combinaciones (incluyendo casos donde las variables entran en conflicto)
casos_soporte = [
    (10, "alta", False),   # Urgencia alta directa -> Nivel 3
    (15, "media", False),  # Estándar rápido -> Nivel 1
    (20, "media", True),   # Urgencia media + Cliente Premium -> Nivel 2
    (50, "media", False),  # Urgencia media + Alto tiempo -> Nivel 2
    (40, "baja", True),    # Urgencia baja + Cliente Premium + Espera > 30 min -> Nivel 2
    (25, "baja", False)    # Urgencia baja + Estándar -> Nivel 1
]

print("--- EVALUACIÓN DE TICKETS DE SOPORTE ---")
for i, (espera, urgencia, premium) in enumerate(casos_soporte, 1):
    accion, motivo = agente_soporte(espera, urgencia, premium)
    print(f"Ticket {i}: Espera={espera}m, Urgencia='{urgencia}', Premium={premium} | Acción: '{accion}' | Motivo: {motivo}")

--- EVALUACIÓN DE TICKETS DE SOPORTE ---
Ticket 1: Espera=10m, Urgencia='alta', Premium=False | Acción: 'escalar a nivel 3' | Motivo: Ticket clasificado con urgencia alta que requiere intervención crítica
Ticket 2: Espera=15m, Urgencia='media', Premium=False | Acción: 'asignar a nivel 1' | Motivo: Urgencia media estándar con tiempo de espera aceptable (15 min)
Ticket 3: Espera=20m, Urgencia='media', Premium=True | Acción: 'escalar a nivel 2' | Motivo: Urgencia media con factor prioritario (Premium=True, Espera=20 min)
Ticket 4: Espera=50m, Urgencia='media', Premium=False | Acción: 'escalar a nivel 2' | Motivo: Urgencia media con factor prioritario (Premium=False, Espera=50 min)
Ticket 5: Espera=40m, Urgencia='baja', Premium=True | Acción: 'escalar a nivel 2' | Motivo: Cliente premium con espera prolongada (40 min) para urgencia baja
Ticket 6: Espera=25m, Urgencia='baja', Premium=False | Acción: 'asignar a nivel 1' | Motivo: Ticket de rutina en tiempo tolerable (25 min)
